In [25]:
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point, MultiPolygon, mapping, box
import json

In [6]:
points = gpd.read_parquet("./data/clean.parquet")[["geo_lat", "geo_lon", "geometry"]]
points = points.drop_duplicates(subset=["geo_lat", "geo_lon"])
points = points.to_crs(32637)
points.head()

,geo_lat,geo_lon,geometry
20,55.738876,37.825370,POINT (426249.603 6177642.656)
29,55.798921,37.738090,POINT (420891.649 6184421.021)
34,55.655307,37.614605,POINT (412831.601 6168587.463)
44,55.701330,37.507412,POINT (406197.791 6173848.674)
56,55.730785,37.631424,POINT (414055.506 6176965.665)


In [8]:
stations = ox.features_from_place("Moscow, Russia", tags={"station": "subway"})

In [9]:
stations["railway"].value_counts(dropna=False)
stations.geom_type.value_counts()      # должны быть в основном Point
stations["name"].nunique()

242

In [22]:
msk = gpd.read_file("./data/moscow_city.geojson")         
area = (msk.to_crs(32637)          
           .buffer(5000)           
           .union_all()            
           .simplify(200))         

area_wgs = gpd.GeoSeries([area], crs=32637).to_crs(4326).iloc[0]

geojson_data = {
    "type": "MultiPolygon",
    "coordinates": mapping(area_wgs)["coordinates"]
}

with open("./data/msk_area_mask.geojson", "w", encoding="utf-8") as file:
    json.dump(geojson_data, file)


In [26]:
# msk_bbox = box(*area_wgs.bounds)
# stations = ox.features_from_polygon(msk_bbox, tags={"station": "subway"})

In [71]:
stations = gpd.read_file("./data/osm/metro.geojson")

In [72]:
stations.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 281 entries, 0 to 280
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   element   281 non-null    str     
 1   name      281 non-null    str     
 2   railway   278 non-null    str     
 3   station   281 non-null    str     
 4   geometry  281 non-null    geometry
dtypes: geometry(1), str(4)
memory usage: 22.1 KB


In [73]:
metro = stations.reset_index()[["geometry", "name"]]
metro.describe()

,geometry,name
count,281,281
unique,281,244
top,POINT (37.66155 55.8871767),Деловой центр
freq,1,3


In [74]:
metro = metro[metro.geom_type == "Point"]
metro = metro.to_crs(32637)

metro_grouped = (
    metro
    .dissolve(
        by="name",
        aggfunc="first",
    )
    .reset_index()
)

metro_grouped["geometry"] = metro_grouped.geometry.centroid

metro_grouped.describe()

,name,geometry
count,244,244
unique,244,244
top,Авиамоторная,POINT (419634.07356757915 6179357.799152892)
freq,1,1


In [79]:
dists = gpd.sjoin_nearest(
    points, 
    metro_grouped, 
    how="left",
    distance_col="distance_metro",
)

dists = dists[["geo_lat", "geo_lon", "distance_metro"]]

dists.describe()

,geo_lat,geo_lon,distance_metro
count,46639.000000,46639.000000,46639.000000
mean,55.735213,37.588813,1572.642113
std,0.105207,0.149964,3336.719796
min,55.179939,36.836900,8.073499
25%,55.670130,37.500436,536.850577
50%,55.743035,37.591158,814.833734
75%,55.807979,37.687951,1254.156594
max,56.011032,37.950668,42645.877856


In [82]:
rail = gpd.read_file("./data/osm/rail.geojson")
rail.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 544 entries, 0 to 543
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   element   544 non-null    str     
 1   name      543 non-null    str     
 2   railway   544 non-null    str     
 3   station   279 non-null    str     
 4   landuse   2 non-null      str     
 5   geometry  544 non-null    geometry
dtypes: geometry(1), str(5)
memory usage: 44.6 KB


In [85]:
rail.describe()

,element,name,railway,station,landuse,geometry
count,544,543,544,279,2,544
unique,2,438,2,2,1,544
top,node,Кунцевская,station,subway,railway,POINT (37.6537207 55.7788343)
freq,542,4,376,278,2,1


In [77]:
estates = estates.merge(
    dists[
        ["geo_lat", "geo_lon", "distance_metro"]
    ],
    on=["geo_lat", "geo_lon"],
    how="left",
)

In [78]:
estates['distance_metro'].isna().sum()

np.int64(0)